# SLM Baseline Evaluation

Evaluates three open-source Small Language Models (SLMs) in a **zero-shot** setting
on the SVD-Benchmark.

| Model | Params | Source |
|-------|--------|--------|
| Qwen2.5-Coder-Instruct | 1.5B | Qwen/Qwen2.5-Coder-1.5B-Instruct |
| CodeGemma-IT | 2B | google/codegemma-2b-it |
| Llama 3.2 Instruct (base) | 1B | unsloth/Llama-3.2-1B-Instruct-bnb-4bit |

Each model receives the **same Alpaca-style zero-shot prompt** used by VulnLocAI
(Table 2 of the paper). No fine-tuning is applied — these are off-the-shelf weights.


In [ ]:
# !pip install transformers accelerate bitsandbytes scikit-learn tqdm --quiet
import torch, pandas as pd, sys
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
sys.path.insert(0, ".")
from metrics_utils import parse_model_output, compute_all_metrics


## 1. Shared Inference Prompt

Identical to VulnLocAI inference prompt (Table 2, paper).

In [ ]:
INSTRUCTION = "Analyze the following Java code snippet and identify any security vulnerability"

ALPACA_PROMPT = """### Instruction:
{instruction}

### Input:
{code_snippet}

### Response:
"""

def build_prompt(code_snippet):
    return ALPACA_PROMPT.format(instruction=INSTRUCTION, code_snippet=code_snippet)


## 2. Inference Helper

Greedy decoding, `max_new_tokens=256`. 4-bit quantisation via BitsAndBytes to match VulnLocAI's memory footprint.

In [ ]:
def load_model(model_id: str):
    bnb_config = BitsAndBytesConfig(load_in_4bit=True,
                                     bnb_4bit_compute_dtype=torch.float16)
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    return model, tokenizer


def predict_slm(code_snippet: str, model, tokenizer,
                max_new_tokens: int = 256, max_length: int = 2048) -> str:
    prompt = build_prompt(code_snippet)
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=max_length).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


## 3. Load SVD-Benchmark

In [ ]:
benchmark_df = pd.read_csv("Evaluation-Benchmark/SVD-Benchmark.csv")
print(f"Loaded {len(benchmark_df)} samples")


## 4. Evaluate All SLM Baselines

SLM model IDs match the architectures listed in Table 4 (SLM specs) of the paper.

In [ ]:
SLM_MODELS = {
    "Qwen2.5-Coder (1.5B)": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "CodeGemma (2B)":        "google/codegemma-2b-it",
    "Llama 3.2 (1B)":        "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
}

all_results = {}

for name, model_id in SLM_MODELS.items():
    print(f"\n{'='*60}")
    print(f"  Evaluating: {name}  ({model_id})")
    print(f"{'='*60}")

    model, tokenizer = load_model(model_id)

    predictions = []
    raw_col = []
    for _, row in tqdm(benchmark_df.iterrows(), total=len(benchmark_df), desc=name):
        raw = predict_slm(str(row["Code Snippet"]), model, tokenizer)
        predictions.append(parse_model_output(raw))
        raw_col.append(raw)

    benchmark_df[f"{name}_raw"]       = raw_col
    benchmark_df[f"{name}_pred_label"]= [p["predicted_label"] for p in predictions]
    benchmark_df[f"{name}_pred_line"] = [p["predicted_line"]  for p in predictions]

    all_results[name] = compute_all_metrics(predictions, benchmark_df, model_name=name)

    # Free GPU memory before loading next model
    del model
    torch.cuda.empty_cache()


## 5. Summary Comparison Table

In [ ]:
summary = pd.DataFrame(list(all_results.values()))
print(summary.to_string(index=False))
summary.to_csv("Results/SLM_baselines_metrics.csv", index=False)
print("\nSaved to Results/SLM_baselines_metrics.csv")
